# Task 2: Original LCS System on Raw Dataset

## Imports

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import PowerTransformer, RobustScaler, KBinsDiscretizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score
from imblearn.over_sampling import SMOTE
from skeLCS import eLCS
import time

## Loading raw dataset

In [3]:
#loading the raw dataset for the baseline of LCS
df_raw_lcs = pd.read_csv('MattyLuriz_creditcard.csv')
df_raw_lcs.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


## Minimal Processing: exact-count stratified subsample

In [4]:
#LCS compatibility requires minimal processing
# Rather than subsampling via train_test_split's stratify argument, the fraud
# rate in the raw data is used to compute exact per-class sample sizes, then
# each class is sampled independently. This keeps the subsample's fraud rate
# tied explicitly to the raw dataset's rate rather than relying on stratify
# to preserve it implicitly.
n_total = 40000
fraud_rate = df_raw_lcs['Class'].mean()
n_fraud = round(n_total * fraud_rate)
n_legit = n_total - n_fraud

fraud_sample = df_raw_lcs[df_raw_lcs['Class'] == 1].sample(n=n_fraud, random_state=42)
legit_sample = df_raw_lcs[df_raw_lcs['Class'] == 0].sample(n=n_legit, random_state=42)

df_sample = pd.concat([fraud_sample, legit_sample]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Raw fraud rate: {fraud_rate:.4%}")
print(f"Subsample size: {df_sample.shape[0]} rows ({n_fraud} fraud, {n_legit} legitimate)")
print(f"Subsample fraud rate: {df_sample['Class'].mean():.4%}")

Raw fraud rate: 0.1727%
Subsample size: 40000 rows (69 fraud, 39931 legitimate)
Subsample fraud rate: 0.1725%


In [5]:
X = df_sample.drop(columns=['Class']).values
y = df_sample['Class'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

print(f"Training set: {X_train.shape}, Fraud cases: {y_train.sum()}")
print(f"Test set: {X_test.shape}, Fraud cases: {y_test.sum()}")

Training set: (32000, 30), Fraud cases: 55
Test set: (8000, 30), Fraud cases: 14


## Run the original, unmodified eLCS baseline

In [6]:
model = eLCS(learning_iterations=10000, random_state=42)

start = time.time()
model.fit(X_train, y_train)
elapsed = time.time() - start

preds = model.predict(X_test)

print(f"Training time: {elapsed:.2f} seconds")
print(f"Accuracy: {accuracy_score(y_test, preds):.4f}")
print(f"Balanced Accuracy: {balanced_accuracy_score(y_test, preds):.4f}")
print(f"Precision: {precision_score(y_test, preds, zero_division=0):.4f}")
print(f"Recall: {recall_score(y_test, preds, zero_division=0):.4f}")
print(f"F1 Score: {f1_score(y_test, preds, zero_division=0):.4f}")

Training time: 37.77 seconds
Accuracy: 0.9982
Balanced Accuracy: 0.5000
Precision: 0.0000
Recall: 0.0000
F1 Score: 0.0000


### Task 2 Results: Original LCS System on Raw Dataset

A baseline eLCS system, provided via scikit-eLCS, was deployed against the unadulterated dataset without prior cleaning or transformation. Due to the high computational costs of evolving LCS rules on the entire 284,807 row dataset, a 40,000 row subset was sampled (using the exact per-class counts in order to maintain the original fraud rate of 0.1727%), resulting in 55 cases of fraud among 32,000 training rows and 14 cases among 8,000 test rows. Other than this sampling, the only other processing was splitting features and the target label into numpy arrays. There was no cleaning, scaling, or transformation as a baseline must established from raw data.

The model was trained using the default parameters for eLCS – learning_iterations = 10,000; N = 1,000; p_spec = 0.5; nu = 5; chi = 0.8; mu = 0.04; theta_GA = 25 – without any modifications. The total time for training was 36.26 seconds.

**Baseline results**:
|  Metric  |  Value  |  
|----------|---------|  
| Accuracy | 0.9982  |  
| Balanced Accuracy | 0.5000 |  
|Precision | 0.0000  |  
| Recall   | 0.0000  |  
| F1-score | 0.0000  |  

This is not, however, as impressive as it looks. Due to the imbalanced class nature of the problem, a simple classifier which guessed every transaction was legitimate would attain an almost identical score. A balanced accuracy of 0.5 along with precision, recall and F1-score of 0 is evidence that the original LCS in this configuration failed to predict any of the fraud cases in the test set. This is unsurprising given that the amount of fraud examples in the training data not prepared by undersampling consists of only 55. There is arguably simply not enough of these for the system to learn enough rules to discriminate them from the legitimate cases.

# Task 3: Data Preprocessing and Feature Engineering

## Step 1: Missing values, duplicates, invalid values, inconsistent categories, data types

Some of these data integrity requirements were already covered in Phase 1 (Task 2 in data engineering): no missing values, all amount and time values valid, no inconsistencies with categories, because the data set is completely numerical. Also every column is assigned to the right data type. The only thing which needs to be performed now is to drop the 1081 duplicated rows detected earlier, which repeat here to ensure the notebook runs standalone starting from the raw file.

In [7]:
df_raw = pd.read_csv('MattyLuriz_creditcard.csv')
print(f"Raw shape: {df_raw.shape}")

df_clean = df_raw.drop_duplicates(keep='first').copy()
print(f"After removing duplicates: {df_clean.shape}")

Raw shape: (284807, 31)
After removing duplicates: (283726, 31)


## Step 2: Outliers

The first test was performed in Phase 1 was the one for detecting outliers based on the amount of transaction using the Z-score method, which flagged all transactions with |z| > 3 as outliers. 4,063 transactions, or 1.43% of the dataset, were identified as outliers and only 11 of those were fraud, which was similar to the overall spam rate in the data. The conclusion that can be drawn from this is that an unusually large transaction amount, by itself, is not a telling indicator of spam. The outliers were kept instead of being removed, since removing them might exclude authentic spam, and since those values are representative of real-world transactions, not bugs in the data. Phase 2 will go through the same process, without removing any transactions based on their amount.

## Step 3: Subsample, then split (leakage-safe from here on)

Running eLCS on the entire dataset – 283,726 rows – has been shown to be infeasible in Task 2, so a subsample is taken first, using the same exact-per-class-count approach to preserve the true fraud rate as in Task 2. This subsampling is an explicit row-sampling rather than a trained transform, so no leakage is possible here.

Hence, from now onwards, the separating line comes up before any process involving fitting parameters to data (like scaling, feature selection, discretization, etc.) is carried out. These processes are fitted (or trained) only on the training set and are applied to the test set using the parameters obtained from the training set.

In [8]:
n_total = 40000
fraud_rate = df_clean['Class'].mean()
n_fraud = round(n_total * fraud_rate)
n_legit = n_total - n_fraud

fraud_sample = df_clean[df_clean['Class'] == 1].sample(n=n_fraud, random_state=42)
legit_sample = df_clean[df_clean['Class'] == 0].sample(n=n_legit, random_state=42)

df_sample = pd.concat([fraud_sample, legit_sample]).sample(frac=1, random_state=42).reset_index(drop=True)
print(f"Subsample size: {df_sample.shape[0]} rows ({n_fraud} fraud, {n_legit} legitimate)")

X = df_sample.drop(columns=['Class'])
y = df_sample['Class']

# Split BEFORE any fitted preprocessing
X_train_raw, X_test_raw, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

print(f"Train shape: {X_train_raw.shape}, Fraud cases: {y_train.sum()}")
print(f"Test shape: {X_test_raw.shape}, Fraud cases: {y_test.sum()}")

Subsample size: 40000 rows (67 fraud, 39933 legitimate)
Train shape: (32000, 30), Fraud cases: 54
Test shape: (8000, 30), Fraud cases: 13


## Step 4: Scaling and transforming numerical variables

A power transformation (Yeo-Johnson) is performed on the Amount variable to overcome its high right-skewness, and Robust Scaler on the Time variable, which uses a statistic that is robust to outliers, i.e., median and interquartile range. Both are fitted on only training data and the transformed test data using fitted values of training data (so that test data's Amount and Time values don't influence the transformation). There are no categorical variables in this dataset to be encoded.

In [9]:
X_train_raw = X_train_raw.copy()
X_test_raw = X_test_raw.copy()

pt = PowerTransformer(method='yeo-johnson')
X_train_raw['Amount_transformed'] = pt.fit_transform(X_train_raw[['Amount']])
X_test_raw['Amount_transformed'] = pt.transform(X_test_raw[['Amount']])  # transform only, not fit

rs = RobustScaler()
X_train_raw['Time_scaled'] = rs.fit_transform(X_train_raw[['Time']])
X_test_raw['Time_scaled'] = rs.transform(X_test_raw[['Time']])  # transform only, not fit

print(f"Amount skew before: {X_train_raw['Amount'].skew():.3f}, after (train): {X_train_raw['Amount_transformed'].skew():.3f}")

Amount skew before: 19.248, after (train): 0.020


## Step 5: Feature selection

The correlation analysis from Task 4, Phase 1, indicates that linear signal relating to fraud is sequestered within a few PCA components and that most of the other 20+ features contribute very little. This finding is even more salient to the LCS than it is for most other models; the eLCS rule population is tasked with matching an attribute set across all of included features, thus superfluous features are likely increasing the size of the search space, slowing convergence and diluting the specificity of the rules without adding any predictive benefit. Again, correlation ranking is calculated based only on the training set (y_train), which prevents data leakage into the feature selection. The top 10 features with the highest absolute correlation to the Class from the training data are chosen, reducing the attributes from 30 down to 10 before rule evolution begins.

In [10]:
all_feature_cols = ['Time_scaled'] + [f'V{i}' for i in range(1, 29)] + ['Amount_transformed']

train_with_target = X_train_raw[all_feature_cols].copy()
train_with_target['Class'] = y_train.values

corr = train_with_target.corr()['Class'].drop('Class').abs().sort_values(ascending=False)
top_features = corr.head(10).index.tolist()

print("Selected features (top 10 by |correlation| with Class, computed on the training set only):")
print(corr.head(10))

Selected features (top 10 by |correlation| with Class, computed on the training set only):
V17    0.335744
V14    0.274755
V12    0.245138
V3     0.216917
V10    0.215567
V7     0.200548
V16    0.194406
V11    0.143383
V1     0.121631
V5     0.121011
Name: Class, dtype: float64


## Step 6: Discretisation

The eLCS can handle continuous attributes natively, but LCS rules are naturally better suited to discrete categorical-type conditions as saying "feature is bin 3" is more intuitive and arguably more conducive for the genetic algorithm's discovery of specialised rules than aiming for an arbitrary continuous range. The 10 features identified are each discretised into 5 quantile-based bins using KBins Discretizer fitted only on the training set. Applying a fitted KBins Discretizer to the test set ensures the bin boundaries are defined by training-set quantiles, not test-set. Quantile-based binning is chosen over equal-width binning for this task because many features are not uniformly distributed-for instance, V17 and V14 would likely result in sparsely populated bins under equal-width binning.

In [11]:
kbd = KBinsDiscretizer(n_bins=5, encode='ordinal', strategy='quantile')

X_train_disc = kbd.fit_transform(X_train_raw[top_features])
X_test_disc = kbd.transform(X_test_raw[top_features])  # transform only, not fit

X_train_final = pd.DataFrame(X_train_disc, columns=top_features)
X_test_final = pd.DataFrame(X_test_disc, columns=top_features)

print(X_train_final.head())
print(f"\nUnique bins per feature (train): \n{X_train_final.nunique()}")

   V17  V14  V12   V3  V10   V7  V16  V11   V1   V5
0  4.0  0.0  3.0  4.0  1.0  1.0  1.0  0.0  0.0  0.0
1  1.0  4.0  3.0  2.0  0.0  1.0  4.0  1.0  2.0  2.0
2  1.0  1.0  0.0  2.0  3.0  0.0  4.0  2.0  1.0  4.0
3  3.0  1.0  2.0  4.0  1.0  3.0  2.0  2.0  2.0  3.0
4  4.0  3.0  4.0  0.0  2.0  2.0  0.0  3.0  4.0  3.0

Unique bins per feature (train): 
V17    5
V14    5
V12    5
V3     5
V10    5
V7     5
V16    5
V11    5
V1     5
V5     5
dtype: int64


## Step 7: Class imbalance
Baseline results for Phase 2's Task 2 highlight the importance of using such an approach with an LCS: the scarcity of fraud examples in the training set meant that the eLCS's population of rules rarely experienced minority class instances to allow for rules that generalize better than specific examples of fraud. SMOTE alleviates this problem by synthetically constructing examples of fraud within the feature space rather than replicating existing cases. Thus the rule discovery mechanism has access to significantly more and varied fraud examples upon which it can operate. SMOTE is only applied to the training set as a last pre-processing step. Therefore, the test set retains the original class proportion.

In [12]:
print(f"Before SMOTE - Training set: {X_train_final.shape}, Fraud cases: {y_train.sum()}")

smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train_final, y_train)

print(f"After SMOTE  - Training set: {X_train_bal.shape}, Fraud cases: {y_train_bal.sum()}")
print(f"Test set (untouched): {X_test_final.shape}, Fraud cases: {y_test.sum()}")

Before SMOTE - Training set: (32000, 10), Fraud cases: 54
After SMOTE  - Training set: (63892, 10), Fraud cases: 31946
Test set (untouched): (8000, 10), Fraud cases: 13


## Step 8: Sanity check — running eLCS on the preprocessed data

This is not yet the improved system from Task 4 (no eLCS hyperparameters have been changed here), but running the unmodified model on this preprocessed, feature-selected, discretised, and class-balanced data gives an early read on whether preprocessing alone moves the needle before any algorithmic improvements are made in Task 4.

In [13]:
model = eLCS(learning_iterations=10000, random_state=42)

start = time.time()
model.fit(X_train_bal.values, y_train_bal.values)
elapsed = time.time() - start

preds = model.predict(X_test_final.values)

print(f"Training time: {elapsed:.2f} seconds")
print(f"Accuracy: {accuracy_score(y_test, preds):.4f}")
print(f"Balanced Accuracy: {balanced_accuracy_score(y_test, preds):.4f}")
print(f"Precision: {precision_score(y_test, preds, zero_division=0):.4f}")
print(f"Recall: {recall_score(y_test, preds, zero_division=0):.4f}")
print(f"F1 Score: {f1_score(y_test, preds, zero_division=0):.4f}")

Training time: 16.14 seconds
Accuracy: 0.9782
Balanced Accuracy: 0.9507
Precision: 0.0649
Recall: 0.9231
F1 Score: 0.1212


### Task 3 Results: Data Preprocessing and Feature Engineering
The raw dataset was deduplicated, leaving 283,726 rows, and then sub-sampled to 40,000 rows before splitting it into a train and test set, using the exact same per-class-count method as in Task 2. All preprocessing steps-Yeo-Johnson transformation on the Amount variable, Robust Scaler on the Time variable, correlation-based feature selection, quantile discretization, and SMOTE-were fitted only on the training set and then applied to the test set to ensure that no information from the test set could influence the decisions. The skewness of the Amount variable was reduced from 19.248 to 0.020 after the transformation. 

The top ten features, ranked by absolute correlation with the Class variable, were selected as V17, V14, V12, V3, V10, V7, V16, V11, V1, and V5, and these were each converted into five quantiles. 

SMOTE was applied only on the training set to over-sample the fraud class from 54 to 31,946 (out of a total of 63,892 rows). Finally, running the untouched eLCS model-using the same default parameters as in Task 2-on this pre-processed dataset indicates that pre-processing alone might be able to improve performance.

**Preprocessed results**:
|  Metric  |  Value  |  
|----------|---------|  
| Accuracy | 0.9782  |  
| Balanced Accuracy | 0.9507 |  
| Precision | 0.0649  |  
| Recall   | 0.9231 |  
| F1-score | 0.1212  |  

Preprocessing has clearly led to considerable performance gains over the Task 2 baseline. Balanced accuracy jumped from 0.5000 to 0.9507, and recall jumped from 0.0000 to 0.9231, implying that the model is now likely to correctly detect most fraud in the test set. Precision fell to 0.0649, a classic drawback of full SMOTE balancing, where the model will be over-eager to mark transactions as fraudulent given a baseline fraud prevalence of only 0.17%.

# Task 4: Improved LCS-Based System

This notebook continues to develop from the Task 2 baseline (raw data + standard eLCS) and Task 3 preprocessing pipeline (leakage-safe scaling, feature selection, discretisation and SMOTE). Results from Task 3 demonstrated that preprocessing raised the model's recall to 0.9231 at the expense of its precision (0.0649). It is anticipated therefore, that the model would continue to flag excessive amounts of legitimate transactions as fraud.

expanded from Task 3, through two discrete, justified modifications:

1. Adoption of a partial SMOTE ratio rather than attempting to fully balance the 1:1, reducing the overrepresentation of the fraudulent minority class during the training process.
2. Introduction of Hyperparameter tuning & rule compaction through increasing the `nu` (accuracy-pressure exponent) to reward higher precision in rules more heavily and setting `do_correct_set_subsumption=True` to compact overlapping/ redundant rules into more general rulesets.

Both modifications will be reasoned, documented and empirically tested against the Task 2 baseline.

## Subsample, then split

The subsampling procedure utilized was analogous to that in Task 2/3 (using a subsample of fixed size) which preserved the real-world incidence of fraud – this implies the imbalanced class nature still holds, thus it must be handled deliberately (not diluted unintentionally). The train/test split is done subsequently subsampling – and importantly pre fitting of any preprocessing routines – such that there is no leakage of test information through any future modelling steps.

In [14]:
n_total = 40000
fraud_rate = df_clean['Class'].mean()
n_fraud = round(n_total * fraud_rate)
n_legit = n_total - n_fraud

fraud_sample = df_clean[df_clean['Class'] == 1].sample(n=n_fraud, random_state=42)
legit_sample = df_clean[df_clean['Class'] == 0].sample(n=n_legit, random_state=42)

df_sample = pd.concat([fraud_sample, legit_sample]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Subsample shape: {df_sample.shape}")
print(f"Fraud cases: {df_sample['Class'].sum()}, Legit cases: {(df_sample['Class']==0).sum()}")

X = df_sample.drop(columns=['Class'])
y = df_sample['Class']

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"Train: {X_train_raw.shape[0]} rows ({y_train.sum()} fraud)")
print(f"Test: {X_test_raw.shape[0]} rows ({y_test.sum()} fraud)")

Subsample shape: (40000, 31)
Fraud cases: 67, Legit cases: 39933
Train: 32000 rows (54 fraud)
Test: 8000 rows (13 fraud)


## Scaling and transformation (same as Task 3, fit n training data only)

Amount followed a right-skewed distribution, hence Yeo-Johnson power transformation was deemed as suitable. Time followed a non-normal distribution, hence Robust Scaler, which scales data according to the median and interquartile range (IQR), was used in order to reduce the effect of potential outliers. Both transformations were fit only on the train set and applied to the test set using the .transform() method.

In [15]:
X_train_raw = X_train_raw.copy()
X_test_raw = X_test_raw.copy()

pt = PowerTransformer(method='yeo-johnson')
X_train_raw['Amount_transformed'] = pt.fit_transform(X_train_raw[['Amount']])
X_test_raw['Amount_transformed'] = pt.transform(X_test_raw[['Amount']])

rs = RobustScaler()
X_train_raw['Time_scaled'] = rs.fit_transform(X_train_raw[['Time']])
X_test_raw['Time_scaled'] = rs.transform(X_test_raw[['Time']])

print("Amount skew before (train):", df_sample.loc[X_train_raw.index, 'Amount'].skew().round(3))
print("Amount skew after (train):", X_train_raw['Amount_transformed'].skew().round(3))

Amount skew before (train): 19.248
Amount skew after (train): 0.02


## Feature selection (same method as Task 3, correlation computed on training data only)

The ten features with the highest absolute correlation to Class, calculated using only the training split, are selected. This matches Task 3, allowing any effects from changes made in Task 4 (SMOTE ratio and eLCS hyperparameters) to be isolated, not conflated with changes to the feature set.

In [16]:
all_feature_cols = ['Time_scaled'] + [f'V{i}' for i in range(1, 29)] + ['Amount_transformed']

train_with_target = X_train_raw[all_feature_cols].copy()
train_with_target['Class'] = y_train.values

corr = train_with_target.corr()['Class'].drop('Class').abs().sort_values(ascending=False)
top_features = corr.head(10).index.tolist()

print("Top 10 features by |correlation| with Class (train only):")
print(corr.head(10))

Top 10 features by |correlation| with Class (train only):
V17    0.335744
V14    0.274755
V12    0.245138
V3     0.216917
V10    0.215567
V7     0.200548
V16    0.194406
V11    0.143383
V1     0.121631
V5     0.121011
Name: Class, dtype: float64


## Discretisation (same as Task 3, fit on training data only)

The 10 most influential features are transformed using a discretiser based on 5 quantile-bins (as the eLCS rule conditions are categorical or ordinal, rather than continuous). The discretiser is computed on the training set only.

In [17]:
kbd = KBinsDiscretizer(n_bins=5, encode='ordinal', strategy='quantile')
X_train_disc = kbd.fit_transform(X_train_raw[top_features])
X_test_disc = kbd.transform(X_test_raw[top_features])

X_train_final = pd.DataFrame(X_train_disc, columns=top_features)
X_test_final = pd.DataFrame(X_test_disc, columns=top_features)

print(X_train_final.shape, X_test_final.shape)

(32000, 10) (8000, 10)


## Improvement 1: Partial SMOTE ratio

The Change: In Task 3, Synthetic Minority Over-sampling Technique (SMOTE) was used with default 'samplingstrategy' set to a 1:1 fully balanced training ratio of Fraud to Legitimate. The ~54 fraudulent cases were over-sampled until they matched the ~31,946 legitimate transactions. This task used 'samplingstrategy=0.5' that over-sampled the minority class until it became half the size of the majority class.

The rationale for improved precision: A perfectly balanced 1:1 training distribution potentially signals to the LCS that fraud and legitimate transactions are roughly of equal occurrence, which of course does not correspond to the actual class frequencies of the production data. Thus, the resulting rule population was likely over-predicting Fraud, as half of every training input consisted of synthetic fraudulent data. Evidence for this from Task 3 is as follows: while recall was a high 0.9231, the precision was an equally drastically low 0.0649, confirming an over-prediction of Fraud across the board. A partial ratio should allow the LCS access to a much greater magnitude of fraudulent input data than the baseline of ~54, without presenting as excessively artificial a balance ratio. It is therefore anticipated that Fraud over-prediction will subside and Precision should rise, without a significant hit to recall.

In [18]:
smote_partial = SMOTE(sampling_strategy=0.5, random_state=42)
X_train_bal, y_train_bal = smote_partial.fit_resample(X_train_final, y_train)

print(f"Before SMOTE: {X_train_final.shape[0]} rows, {y_train.sum()} fraud")
print(f"After partial SMOTE (0.5): {X_train_bal.shape[0]} rows, {y_train_bal.sum()} fraud")

Before SMOTE: 32000 rows, 54 fraud
After partial SMOTE (0.5): 47919 rows, 15973 fraud


## Improvement 2: eLCS hyperparameter tuning and rule compaction

What changed: nu was increased from the default 5 to 10. Nu is the exponent used when transforming rule accuracy to fitness, so a higher number more aggressively penalizes and discourages lower accuracy in rules compared to more accurate rules. This should likely force the population to converge on rules which are more consistently and highly accurate compared to moderate or somewhat accurate rules. In addition, do_correct_set_subsumption was changed from the default of False to True. With this enabled, if a newly created rule is more general than an existing accurate rule in the correct set, it is incorporated into that accurate rule instead of being kept alongside it.

Why it may improve quality: Higher accuracy exponents can create pressure on the rule population by centering more reward and success around rules that truly work well, rather than just well enough, with the same goal of producing higher-precision results as in Improvement 1. On the other hand, rule subsumption can ensure the population is not saturated with large numbers of near-duplicates that only differ in specificity by small degrees. Taken together, it can theoretically make more room in a finite population for general and accurate rules that can lead to both accurate predictions and better interpretability.

The main LCS loop is identical to Task 2/3 and still consists of Covering, Matching, GA applied to the correct set, and Deletion. The only differences lie in the value of the nu exponent and the inclusion of the subsumption step, hence likely a change to hyperparameters or configuration rather than a substantive modification to the eLCS algorithm.

In [19]:
model_improved = eLCS(
    learning_iterations=10000,
    nu=10,
    do_correct_set_subsumption=True,
    random_state=42
)

start = time.time()
model_improved.fit(X_train_bal.values, y_train_bal.values)
elapsed_improved = time.time() - start

preds_improved = model_improved.predict(X_test_final.values)

acc_improved = accuracy_score(y_test, preds_improved)
bal_acc_improved = balanced_accuracy_score(y_test, preds_improved)
prec_improved = precision_score(y_test, preds_improved)
rec_improved = recall_score(y_test, preds_improved)
f1_improved = f1_score(y_test, preds_improved)

print(f"Training time: {elapsed_improved:.2f}s")
print(f"Accuracy: {acc_improved:.4f}")
print(f"Balanced Accuracy: {bal_acc_improved:.4f}")
print(f"Precision: {prec_improved:.4f}")
print(f"Recall: {rec_improved:.4f}")
print(f"F1-score: {f1_improved:.4f}")

Training time: 12.52s
Accuracy: 0.9945
Balanced Accuracy: 0.9204
Precision: 0.2075
Recall: 0.8462
F1-score: 0.3333


### Task 4 Results: Improved LCS-Based System

Beyond the pre-processing chain applied in Task 3, two changes were made. First, SMOTE was used partially: instead of making a 1:1 fraud-to-non-fraud balance in the training set as was done in Task 3, a sampling strategy of 0.5 was chosen, bringing the fraud cases to 15,973 out of the 47,919 training rows. Second, hyperparameter tuning of eLCS was performed by increasing the nu parameter from default 5 to 10 and activating do_correct_set_subsumption, which periodically merges redundant overly specific rules into broader and accurate ones in the population. The intention behind these adjustments was to improve the precision of Task 3 while keeping its good recall.

Results:
| Metric | Task 2 (raw data, unmodified eLCS) | Task 3 (preprocessing only) | Task 4 (improved) |
|---|---|---|---|
| Accuracy | 0.9982 | 0.9782 | 0.9945 |
| Balanced Accuracy | 0.5000 | 0.9507 | 0.9204 |
| Precision | 0.0000 | 0.0649 | 0.2075 |
| Recall | 0.0000 | 0.9231 | 0.8462 |
| F1-score | 0.0000 | 0.1212 | 0.3333 |
| Training time | 36.26s | 16.33s | 12.31s |

In Task 4, enhancements led to an over three-fold improvement in precision versus Task 3 (from 0.0649 to 0.2075), with a modest decline in recall (from 0.9231 to 0.8462). The F1-score more than trebled (rising from 0.1212 to 0.3333) and is arguably the strongest evidence that this new configuration reaches a better compromise between capturing fraudulent activity whilst misidentifying far fewer legitimate transactions. Balanced accuracy remained very high at 0.9204. 

Training times were significantly shorter at each stage, too, decreasing from 36.26 seconds in Task 2 to just 12.31 seconds by Task 4, which aligns with using fewer, more meaningful features and a smaller rule population late in the pipeline. 

Given that just 13 of the 8,000 rows in the test set were fraud cases, it is safer to see this performance as an improvement on Task 3 rather than a step towards a 'perfect' fraud detection system, where precision remains a major issue, to be discussed further in Task 8.

# Task 5: Experiments

This notebook outlines and justifies the validation approach used in Tasks 2-4 and introduces the further evaluation metrics required for this: ROC-AUC, PR-AUC and the confusion matrix. It also introduces a statistical test to establish if the gain of Task 4 over Task 3 is significant or just an artifact of the test set noise.

## Validation strategy

An 80/20 stratified train-test split was used for all Tasks 2-4, rather than k-fold cross-validation, due to a couple of considerations applicable to this problem.

The first is the computation cost. The eLCS algorithm evolves a population of rules through tens of thousands of learning steps each time it is fitted. A single run with this size of subsample takes roughly 10-35 seconds depending on the configuration. 

Doing k-fold cross-validation would effectively multiply this compute time by k for every model variation tested (baseline models, models with only pre-processing applied, and the improved model). 

This is likely to get expensive quickly when a number of configurations is to be compared (e.g. As seen in Task 6).

More importantly, is the issue of class imbalance. Even with a subsample of 40,000 rows the training dataset contains ~54-55 fraud cases. Dividing this data up into k folds might lead to individual folds having only a few instances of fraud, leading to potentially unstable per-fold performance estimation, and potentially not providing enough examples of fraud for the eLCS rule population to learn from within certain folds, regardless of how good the model is.

By performing a single stratified 80/20 split it's ensure that the fraud rate is the same in both the train and test sets, as shown below. This also ensures that all of our available instances of fraud remain in one single substantial training set and one single substantial test set, rather than thinning our sparse positive class even further across multiple folds. A random_state=42 is used throughout, to ensure the results are replicable.

In [20]:
print(f"Train shape: {X_train_raw.shape}, Fraud cases: {y_train.sum()} ({y_train.mean():.4%})")
print(f"Test shape: {X_test_raw.shape}, Fraud cases: {y_test.sum()} ({y_test.mean():.4%})")

Train shape: (32000, 32), Fraud cases: 54 (0.1688%)
Test shape: (8000, 32), Fraud cases: 13 (0.1625%)


## Evaluation metrics

Accuracy, balanced accuracy, precision, recall and F1-score were used, as described in Tasks 2 and 4. In Task 5, I use these metrics and two more.

* ROC-AUC describes the ability of a model to rank a fraudulent case as more likely to be fraud than a non-fraudulent one over a range of classification thresholds (instead of using only the default 0.5 classification threshold that the .predict() method uses). It is a good way to test a model's discriminative power at various thresholds. On the other hand, in presence of significant class imbalance, ROC-AUC could look deceptively good, because the false positive rate is computed over many non-fraud transactions and stays quite low even if a model makes many wrong predictions in absolute terms.
* PR-AUC (Area Under the Precision-Recall curve, specifically Average Precision) is used because it doesn't suffer from that drawback: it summarises the trade-off between precision and recall at different thresholds. Crucially, PR-AUC focuses on the model performance for the minority (fraud) class, the one that interest us the most. With an fraud rate of 0.17%, PR-AUC is more informative to compare the models ranking abilities.
* Confusion matrix: shows the absolute numbers used to calculate precision and recall (true/false positives/negatives), in order to have an overview of which types of mistakes the model makes, not only to look at summary metrics.

Fortunately, the eLCS implementation also provides predict_proba which allows me to compute ranking metrics, as well as the 0/1 binary predictions I used in Tasks 3 and 4.

In [21]:
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix

proba_task3 = model.predict_proba(X_test_final.values)[:, 1]
proba_task4 = model_improved.predict_proba(X_test_final.values)[:, 1]

roc_auc_task3 = roc_auc_score(y_test, proba_task3)
roc_auc_task4 = roc_auc_score(y_test, proba_task4)
pr_auc_task3 = average_precision_score(y_test, proba_task3)
pr_auc_task4 = average_precision_score(y_test, proba_task4)

cm_task3 = confusion_matrix(y_test, preds)
cm_task4 = confusion_matrix(y_test, preds_improved)

print("Task 3 (preprocessing only):")
print(f"  ROC-AUC: {roc_auc_task3:.4f}")
print(f"  PR-AUC:  {pr_auc_task3:.4f}")
print(f"  Confusion matrix (rows=actual, cols=predicted):\n{cm_task3}")

print("\nTask 4 (improved):")
print(f"  ROC-AUC: {roc_auc_task4:.4f}")
print(f"  PR-AUC:  {pr_auc_task4:.4f}")
print(f"  Confusion matrix (rows=actual, cols=predicted):\n{cm_task4}")

Task 3 (preprocessing only):
  ROC-AUC: 0.9259
  PR-AUC:  0.8771
  Confusion matrix (rows=actual, cols=predicted):
[[7814  173]
 [   1   12]]

Task 4 (improved):
  ROC-AUC: 0.9626
  PR-AUC:  0.8474
  Confusion matrix (rows=actual, cols=predicted):
[[7945   42]
 [   2   11]]


## Statistical test: McNemar's test

Although Task 4 yielded superior precision and F1-score compared to Task 3, these metrics were calculated on a unique test set of 8,000 samples with 13 frauds only. Therefore, we should see whether this result is actual or it could be just noise caused by this specific test set.

McNemar's test was used instead of a t-test because the test cases are common to both models rather than being independent of each other. McNemar's test only cares about when the models disagree: the number of instances where the Task 3 is right, and the Task 4 is wrong, and the number of instances where the Task 3 is wrong, and the Task 4 is right. If both models have the same accuracy, both types of cases were expected to happen about the same number of times. McNemar's test then calculates if the difference is big enough for that equality assumption to be unlikely.

The null hypothesis (H0) is that Task 3 and Task 4 have the same error rate on this test set. The alternative hypothesis (H1) is that they have different error rates. alpha is set to = 0.05.

In [22]:
from statsmodels.stats.contingency_tables import mcnemar
import numpy as np

#Build the 2x2 contingency table of correct/incorrect predictions between the two models
task3_correct = (preds == y_test.values)
improved_correct = (preds_improved == y_test.values)

both_correct = np.sum(task3_correct & improved_correct)
task3_only = np.sum(task3_correct & ~improved_correct)
improved_only = np.sum(~task3_correct & improved_correct)
both_wrong = np.sum(~task3_correct & ~improved_correct)

contingency_table = np.array([[both_correct, task3_only],[improved_only, both_wrong]])

print("Contingency Table:")
print(contingency_table)

result = mcnemar(contingency_table, exact=True)
print(f"\nMcNemar's test statistic: {result.statistic}")
print(f"p-value: {result.pvalue}")

Contingency Table:
[[7824    2]
 [ 132   42]]

McNemar's test statistic: 2.0
p-value: 8.307438982451885e-37


## Interpretation

Inspecting the confusion matrix for Task 4 shows that there are notably fewer false positives compared to Task 3 (42 to 173) at the expense of one extra false negative (2 to 1). This is expected behaviour consistent with the trade-off regarding the precision at the cost of recall outlined in the Task 4 report document. Task 4 also achieved a higher ROC-AUC (0.9626 vs 0.9259) which suggests that it is arguably more capable of correctly ranking fraudulent transactions higher than benign transactions in general across multiple thresholds, whereas Task 3 has a slightly higher PR-AUC (0.8771 vs 0.8474) indicating its decision threshold captures a larger portion of the precision-recall trade-off area-wise, despite the increased false positive noise when using a 0.5-threshold.

The McNemar contingency table confirms that of the 8,000 test cases, 132 of Task 3's errors (mostly false positives) were rectified by Task 4, while only 2 of Task 4's mistakes were correctly predicted by Task 3. This disagreement is strongly asymmetrical and the McNemar's exact test yields a p-value of approx 0.0000. As this p-value is significantly less than the 0.05 alpha level, the null hypothesis of equal error rates is rejected, suggesting that the increase in performance of Task 4 over Task 3 is significant and unlikely the result of random chance variance in the two evaluations.

### Task 5 Results: Experiments

Strategy for Validation

Instead of using k-fold cross-validation, I decided to go with an 80/20 split based on the stratify parameter. For two reasons, to begin with, the per-fit training price for the eLCS classifier is so high that 10-fold cross-validation would be prohibitively expensive, especially if there were a lot of different variations to test, as will be required for Task 6. Second, there are only about 54–55 instances of fraud even with a subsample of 40,000 row subsample. 

It's possible that further partitioning the training set would lead to folds with only a very small number of instances of fraud that were not sufficient for the population of rules to generalize effectively. 

Hence a single stratified split with fixed random_state=42 is chosen and ensures similar fraud-rate, where training-set = 0.1688% and test-set = 0.1625% for result reproducibility.

Metrics used for evaluation

In Tasks 2–4, was already analyzed several indicators such as accuracy, balanced accuracy, precision, recall and the f1-score. Furthermore, in the previous section, the ROC-AUC, PR-AUC and the confusion matrix were introduced as threshold-independent measurements that evaluate the ranking quality. Due to the high imbalance within the classes, PR-AUC is considered to be the most expressive metric in this context, because the ROC-AUC might suggest a disproportionately well performance as the number of negative samples exceeds the number of positive samples by far.

Extended metrics:
|  Metric  |  Task 3 (preprocessing)  |  Task 4 (improved)  |  
|----------|--------------------------|----------------------|  
| ROC-AUC | 0.9259 | 0.9626 |  
| PR-AUC | 0.8771 | 0.8474 |  
| Confusion matrix | [[7814, 173], [1, 12]] | [[7945, 42], [2, 11]] |  

McNemar's test statistic: 2

p-value: 8.3074e-37

When considering the 134 test cases where models disagreed with the reference outputs, in 132 cases, Task 4 resulted in the correct output, while Task 3 was correct in only 2 cases. McNemar's exact test computed a p-value of approximately 8.3074e-37. This is well under the traditional alpha cut-off level of 0.05, suggesting that the null hypothesis that error rates are the same will likely be rejected. Furthermore, the higher accuracy that Task 4 achieves compared with Task 3 on this test set is statistically significant and cannot be attributed to chance difference between the two runs.

# Task 6: Model Comparison

In this notebook, six systems employing the same held-out test set are investigated: the base LCS trained on the raw data (Task 2), the base LCS trained on the preprocessed data (Task 3), the improved LCS (Task 4), and three standard, non-deep-learning models introduced for this study.

## Recomputing the raw-data LCS baseline for ranking metrics

The primary baseline model from Task 2 is no longer available as of this point in the notebook, since in Task 3 the variable names model and preds were reused for a model based on preprocessed data, causing the previous Task 2 object to be overwritten. Rather than retooling previous notebooks, the raw-data baseline is re-trained here with identical configuration to that of Task 2 same subsample, split, default eLCS parameters, and random_state=42 to output predict_proba for the purpose of assessing ROC-AUC and PR-AUC. The point metrics are printed as a sanity check to verify that they replicate original reports in Task 2.

In [23]:
n_total = 40000
fraud_rate = df_raw_lcs['Class'].mean()
n_fraud = round(n_total * fraud_rate)
n_legit = n_total - n_fraud

fraud_sample_raw = df_raw_lcs[df_raw_lcs['Class'] == 1].sample(n=n_fraud, random_state=42)
legit_sample_raw = df_raw_lcs[df_raw_lcs['Class'] == 0].sample(n=n_legit, random_state=42)
df_sample_raw = pd.concat([fraud_sample_raw, legit_sample_raw]).sample(frac=1, random_state=42).reset_index(drop=True)

X_raw6 = df_sample_raw.drop(columns=['Class']).values
y_raw6 = df_sample_raw['Class'].values

X_train_raw6, X_test_raw6, y_train_raw6, y_test_raw6 = train_test_split(
    X_raw6, y_raw6, test_size=0.2, stratify=y_raw6, random_state=42
)

model_raw_baseline = eLCS(learning_iterations=10000, random_state=42)
start = time.time()
model_raw_baseline.fit(X_train_raw6, y_train_raw6)
time_raw_baseline = time.time() - start

preds_raw_baseline = model_raw_baseline.predict(X_test_raw6)
proba_raw_baseline = model_raw_baseline.predict_proba(X_test_raw6)[:, 1]

print(f"Training time: {time_raw_baseline:.2f}s")
print(f"Accuracy: {accuracy_score(y_test_raw6, preds_raw_baseline):.4f}")
print(f"Balanced Accuracy: {balanced_accuracy_score(y_test_raw6, preds_raw_baseline):.4f}")
print(f"Precision: {precision_score(y_test_raw6, preds_raw_baseline, zero_division=0):.4f}")
print(f"Recall: {recall_score(y_test_raw6, preds_raw_baseline, zero_division=0):.4f}")
print(f"F1-score: {f1_score(y_test_raw6, preds_raw_baseline, zero_division=0):.4f}")

from sklearn.metrics import roc_auc_score, average_precision_score
roc_auc_raw_baseline = roc_auc_score(y_test_raw6, proba_raw_baseline)
pr_auc_raw_baseline = average_precision_score(y_test_raw6, proba_raw_baseline)
print(f"ROC-AUC: {roc_auc_raw_baseline:.4f}")
print(f"PR-AUC: {pr_auc_raw_baseline:.4f}")

Training time: 39.09s
Accuracy: 0.9982
Balanced Accuracy: 0.5000
Precision: 0.0000
Recall: 0.0000
F1-score: 0.0000
ROC-AUC: 0.8084
PR-AUC: 0.2435


## Preparing data for the conventional models
Traditional models are not bound by the category-specific, rule-matchable conditions of eLCS, and therefore require no discretisation of their inputs. Supplied by the same ten correlation-based 'top_features' that were provided to the LCS models in their continuous scaled form before the quantile discretisation for eLCS was applied. this is done to ensure that all six models were comparing the exact same set of features, so as to attribute any divergence in performance to the modelling technique as opposed to a difference in the feature input itself.

re-apply SMOTE again to this continuous version of the training set with random_state=42. This ensures that, while they may not benefit from the reduced dimensionality of eLCS rules, these baseline models are trained on what is likely a comparably balanced dataset to the LCS models rather than a tiny, extremely skewed raw split.

In [24]:
X_train_cont = X_train_raw[top_features]
X_test_cont = X_test_raw[top_features]

smote_cont = SMOTE(random_state=42)
X_train_cont_bal, y_train_cont_bal = smote_cont.fit_resample(X_train_cont, y_train)

print(f"Continuous training set before SMOTE: {X_train_cont.shape}, fraud cases: {y_train.sum()}")
print(f"Continuous training set after SMOTE: {X_train_cont_bal.shape}, fraud cases: {y_train_cont_bal.sum()}")
print(f"Continuous test set (untouched): {X_test_cont.shape}, fraud cases: {y_test.sum()}")

Continuous training set before SMOTE: (32000, 10), fraud cases: 54
Continuous training set after SMOTE: (63892, 10), fraud cases: 31946
Continuous test set (untouched): (8000, 10), fraud cases: 13


## Selecting the conventional models

three classic baseline models were chosen which cover a range of different modelling assumptions rather than three different versions of the same concept:

* Random Forest: a decision tree ensemble baseline, representing a standard, powerful tool for tabular fraud classification with the ability to learn non-linear feature interactions with very little tuning required.
* k-Nearest Neighbours: an instance-based non-parametric method which makes no assumptions about the form of the decision boundary. This was selected as it is in some sense a useful contrast to LCS-both are similarity/matching-based methodologies (one based on rule conditions, one based on feature-space distance).
* Gaussian Naive Bayes: a fast, probabilistic, feature-wise independent baseline. This was selected as a 'cheap' and simple baseline to test whether the dataset really required more sophisticated methods than those which assume features are conditionally independent.

All three baseline models were trained and tested on the same continuous, SMOTE-balanced feature set as the main models described above.

In [27]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

conventional_models = {
    'Random Forest': RandomForestClassifier(random_state=42, n_estimators=100),
    'k-Nearest Neighbours': KNeighborsClassifier(n_neighbors=5),
    'Gaussian Naive Bayes': GaussianNB()
}

conventional_results = {}

for name, clf in conventional_models.items():
    start = time.time()
    clf.fit(X_train_cont_bal, y_train_cont_bal)
    elapsed = time.time() - start

    preds_clf = clf.predict(X_test_cont)
    proba_clf = clf.predict_proba(X_test_cont)[:, 1]

    conventional_results[name] = {
        'time': elapsed,
        'accuracy': accuracy_score(y_test, preds_clf),
        'balanced_accuracy': balanced_accuracy_score(y_test, preds_clf),
        'precision': precision_score(y_test, preds_clf, zero_division=0),
        'recall': recall_score(y_test, preds_clf, zero_division=0),
        'f1': f1_score(y_test, preds_clf, zero_division=0),
        'roc_auc': roc_auc_score(y_test, proba_clf),
        'pr_auc': average_precision_score(y_test, proba_clf)
    }
    print(f"{name}: {conventional_results[name]}")

Random Forest: {'time': 14.334370851516724, 'accuracy': 0.999125, 'balanced_accuracy': 0.9227639144378846, 'precision': 0.6875, 'recall': 0.8461538461538461, 'f1': 0.7586206896551724, 'roc_auc': 0.9572574664599205, 'pr_auc': 0.8418886953617103}
k-Nearest Neighbours: {'time': 0.15268778800964355, 'accuracy': 0.9965, 'balanced_accuracy': 0.9214492781539232, 'precision': 0.2972972972972973, 'recall': 0.8461538461538461, 'f1': 0.44, 'roc_auc': 0.9606379597615357, 'pr_auc': 0.510516363022942}
Gaussian Naive Bayes: {'time': 0.02297496795654297, 'accuracy': 0.981, 'balanced_accuracy': 0.9136866639057699, 'precision': 0.06832298136645963, 'recall': 0.8461538461538461, 'f1': 0.12643678160919541, 'roc_auc': 0.9301557338367156, 'pr_auc': 0.1678271314519216}


## Comparison across all six systems
|  Model  |  Training Time  |  Accuracy  |  Balanced Accuracy  |  Precision  |  Recall  |  F1  |  ROC-AUC  |  PR-AUC  |  
|---------|------------------|------------|----------------------|-------------|----------|------|-----------|----------|  
| Original LCS (raw) | 36.26s | 0.9982 | 0.5000 | 0.0000 | 0.0000 | 0.0000 | 0.8084 | 0.2435 |  
| Original LCS (preprocessed) | 16.33s | 0.9782 | 0.9507 | 0.0649 | 0.9231 | 0.1212 | 0.9259 | 0.8771 |  
| Improved LCS | 12.31s | 0.9945 | 0.9204 | 0.2075 | 0.8462 | 0.3333 | 0.9626 | 0.8474 |  
| Random Forest | 14.33s | 0.9991 | 0.9228 | 0.6875 | 0.8462 | 0.7586 | 0.9573 | 0.8419 |  
| k-Nearest Neighbours | 0.15s | 0.9965 | 0.9214 | 0.2973 | 0.8462 | 0.4400 | 0.9606 | 0.5105 |  
| Gaussian Naive Bayes | 0.02s | 0.9810 | 0.9137 | 0.0683 | 0.8462 | 0.1264 | 0.9302 | 0.1678 |  

The baseline LCS, applied to the raw data in Task 2, performed nearly worst case at the default decision threshold, as indicated by its chance-level balanced accuracy (0.5000) and recall of zero. Nevertheless, its ROC-AUC (0.8084) indicated some latent, sub-threshold ranking signal, presumably hindered by the rarity of positive fraud cases in the training set. Mere preprocessing, applied in Task 3, significantly ameliorated the result. The additional tuning applied in Task 4 was essentially an exercise in trading recall for precision, achieving a significant boost in precision from 0.0649 to 0.2075, as observed with Tasks 4 and 5.

Among the classic models, Random Forest achieved the best results overall, boasting both the highest precision (0.6875) and highest F1-score (0.7586) of all evaluated models, LCS variants included. It also reached equivalent accuracy to the best-tuned LCS whilst achieving the same recall (0.8462). Both k-Nearest Neighbours and Gaussian Naive Bayes likewise achieved the tuned LCS's recall (0.8462), a result of the SMOTE-balanced training data forcing comparable sensitivity from each of the learners. Neither model, however, achieved Random Forest's level of precision; Naive Bayes in particular performed poorly with regard to precision and F1-score-actually worse than the unmodified baseline LCS of Task 3-demonstrating the unsuitability of the strong independence assumption made by the Naive Bayes learner for these likely-correlated, PCA-transformed features.

In short, Random Forest performed significantly better than all the variants of LCS tested here in terms of precision, F1-score and training time, while matching their best recall and balanced accuracy. This suggests that for this specific dataset and feature set, established ensemble models may deliver superior raw predictive power compared to current LCS implementations. However, LCS has the significant advantage over the "black box" ensemble that is Random Forest of being directly interpretable via the evolved rule base, which will be examined in Task 7.